# Project 1 – Predictive Task Definition, Data Preparation & Exploratory Analysis

## 1. Predictive Task Definition
## 2. Data Understanding and General Exploration
## 3. Preprocessing Strategy
## 4. Task-Driven Exploratory Data Analysis
## 5. Limitations, Risks, and Assumptions

### 1. Predictive Task Definition

#### Objective

The purpose of this project is to evaluate whether the provided online retail dataset can support a realistic and meaningful predictive task. After examining the structure of the data and the available variables, the task defined for this analysis is:

**To predict whether a delivered order will receive a low customer review score (≤ 2 stars).**

This definition is based on the availability of review scores and the business relevance of customer dissatisfaction.


#### Prediction Timing

For the predictive task to be realistic, it is important to clearly define when the prediction would be made in practice. In this case, the prediction is assumed to take place at the moment the order is handed over to the carrier for delivery.

At that point in time, the following information is already known:
- Order details (purchase timestamp, order value, number of items)
- Payment information (installments, payment amount)
- Product characteristics
- Seller information
- Estimated delivery date

However, any information that becomes available after delivery — such as the actual review score, review timestamps, or review text — must not be used as input features. Including such variables would introduce data leakage and make the predictive task unrealistic.


#### Business Motivation

From a business perspective, predicting potentially low reviews before the customer submits feedback could be highly valuable. If high-risk orders are identified early, the company could:

- Proactively contact customers
- Offer compensation or support
- Monitor specific sellers more closely
- Improve logistics or delivery processes

Therefore, this predictive task is both practically meaningful and supported by the structure of the dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")

In [ ]:
orders = pd.read_csv("orders.csv")
order_items = pd.read_csv("order_items.csv")
order_payments = pd.read_csv("order_payments.csv")
order_reviews = pd.read_csv("order_review.csv")
products = pd.read_csv("products.csv")
product_translation = pd.read_csv("product_category_name_translation.csv")
customers = pd.read_csv("customers.csv")
sellers = pd.read_csv("sellers.csv")

### Initial Data Inspection

After loading the tables, it becomes clear that the dataset follows a relational structure rather than a single flat format. The orders table appears to be the central entity, while other tables provide additional information at different levels of granularity.

In particular:

- `order_items` and `order_payments` contain multiple rows per order, indicating one-to-many relationships.
- `order_review` contains customer feedback that occurs after delivery.
- `products`, `customers`, and `sellers` provide contextual information that can potentially be used as predictors.

This confirms that constructing a single modeling-ready dataset will require careful merging and aggregation at the order level.

In [ ]:
import os
os.listdir()

In [ ]:
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", order_payments.shape)
print("Reviews:", order_reviews.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Sellers:", sellers.shape)

### Dataset Structure

The dataset is organized as multiple interconnected tables rather than a single flat file. Each table represents a different entity within the online retail platform.

The **orders** table acts as the central transaction record, containing information about each purchase.  
The **order_items** table links products and sellers to individual orders, creating a one-to-many relationship, since one order may contain multiple items.  
The **order_payments** table stores payment information, where an order can also have multiple payment records.  
The **order_review** table contains customer feedback submitted after delivery.  
The **products** table provides product-level information such as category and physical characteristics.  
Finally, the **customers** and **sellers** tables include geographic and identification-related information for both sides of the transaction.

Because of this relational structure, the data cannot be used directly for prediction at the order level. Several tables must first be merged and aggregated in order to construct a clean, order-level dataset suitable for further analysis.

In [ ]:
orders = orders[orders["order_status"] == "delivered"]
orders.shape

Since the objective is to predict low review scores, only orders with the status "delivered" are retained. Orders that were canceled or not delivered cannot logically receive a review and would therefore not contribute to the defined predictive task.

Restricting the dataset at this stage ensures consistency between the target variable and the operational process being modeled.

In [ ]:
data = orders.merge(order_reviews, on="order_id", how="inner")
data.shape

The reviews table is merged with the delivered orders to obtain the review score associated with each transaction. An inner join is used to ensure that only orders with available review information are retained.

At this stage, it is important to note that review-related timestamps and textual comments will later be excluded from the feature set, as they occur after delivery and would introduce data leakage.

In [ ]:
data["low_review"] = data["review_score"].apply(lambda x: 1 if x <= 2 else 0)

data["low_review"].value_counts()

To transform the problem into a binary predictive task, a new variable `low_review` is created.
    
Orders with a review score of 1 or 2 are labeled as 1 (low review), while orders with scores above 2 are labeled as 0.
    
- 1 = review score ≤ 2
- 0 = review score > 2

This threshold is chosen to represent clear dissatisfaction, while avoiding ambiguity associated with neutral ratings. Converting the score into a binary variable simplifies the predictive framing and aligns with a practical business objective: identifying clearly negative customer experiences.

In [ ]:
items_agg = order_items.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "product_id": "count",
    "seller_id": "nunique"
}).reset_index()

items_agg.columns = [
    "order_id",
    "total_price",
    "total_freight",
    "num_items",
    "num_sellers"
]

data = data.merge(items_agg, on="order_id", how="left")

Because one order may contain multiple items, aggregation is necessary to construct order-level predictors.

The following aggregated features are created:

- `total_price`: sum of all item prices in the order  
- `total_freight`: total shipping cost  
- `num_items`: number of products purchased  
- `num_sellers`: number of distinct sellers involved  

This aggregation preserves relevant information while ensuring that each row represents a single order.

In [ ]:
payments_agg = order_payments.groupby("order_id").agg({
    "payment_value": "sum",
    "payment_installments": "max"
}).reset_index()

data = data.merge(payments_agg, on="order_id", how="left")

Payment information is also aggregated at the order level. The total payment value is calculated as the sum of all payments associated with an order. The maximum number of installments is retained to capture payment complexity.

These features may reflect purchasing behavior and financial commitment, which could potentially relate to customer satisfaction.

In [ ]:
products = products.merge(product_translation,
                          on="product_category_name",
                          how="left")

product_category_agg = order_items.merge(products,
                                         on="product_id",
                                         how="left")

product_category_agg = product_category_agg.groupby("order_id")[
    "product_category_name_english"
].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else "Unknown").reset_index()

data = data.merge(product_category_agg,
                  on="order_id",
                  how="left")

Product category information is merged to enrich the dataset with semantic product attributes.

Since an order may contain products from multiple categories, the most frequent category (mode) within each order is selected as a representative feature. While this simplification may hide some complexity, it allows category-level analysis at the order level.

In [ ]:
data = data.drop(columns=[
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
])

Review comment text and review timestamps are removed from the dataset because they are recorded after delivery and directly reflect the outcome of interest.

Including these variables as predictors would introduce data leakage, as they contain information that would not be available at the time the prediction is assumed to be made. Preventing leakage is essential to ensure that the predictive task remains realistic and valid.

In [ ]:
numeric_cols = data.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
   data[col]= data[col].fillna(data[col].median(), inplace=True)

data.fillna("Unknown", inplace=True)

Missing numerical values are replaced with the median of their respective columns. The median is chosen instead of the mean to reduce sensitivity to extreme values.

Categorical missing values are replaced with the label "Unknown". This preserves the information that data was missing, rather than discarding rows, which helps maintain the dataset size and avoids unintended bias introduced by deletion.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="low_review", data=data)
plt.title("Distribution of Low Reviews")
plt.show()

The distribution of the target variable shows that low reviews represent a minority of observations. This suggests class imbalance, which would need to be addressed in a future modeling stage. However, for the current exploratory phase, it confirms that dissatisfied customers are present in sufficient numbers to justify the predictive task.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="low_review", y="total_price", data=data)
plt.title("Total Price vs Low Review")
plt.show()

The boxplot indicates whether higher or lower order values are associated with dissatisfaction. If a visible difference exists between the groups, the total price may serve as a meaningful predictor. Even if differences are modest, this exploration helps evaluate feasibility.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="low_review", y="num_items", data=data)
plt.title("Number of Items vs Low Review")
plt.show()

The relationship between the number of items and low reviews may reveal whether complex orders are more prone to dissatisfaction. Larger orders may introduce higher logistical risk.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="low_review", y="payment_installments", data=data)
plt.title("Installments vs Low Review")
plt.show()

Payment installments may reflect purchasing behavior or financial strain. Observing differences between satisfied and dissatisfied customers provides insight into whether payment structure contributes to review outcomes.

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(y="product_category_name_english",
              hue="low_review",
              data=data,
              order=data["product_category_name_english"].value_counts().index[:10])
plt.title("Top Categories vs Low Review")
plt.show()

Product categories may vary in quality, shipping complexity, or customer expectations. If certain categories show disproportionately higher low-review counts, this strengthens the case for category-level predictors.

 ### Limitations, Risks, and Assumptions

Several limitations should be acknowledged:

- Review scores are subjective and may vary based on individual customer expectations.
- Only customers who submitted reviews are included, which may introduce self-selection bias.
- Aggregating item-level information to the order level may obscure product-specific issues.
- Some operational factors (e.g., internal logistics decisions) are not observable in the dataset.
- The prediction timing assumption (at carrier handover) simplifies real-world operational complexity.

These limitations highlight that while the predictive task appears feasible, careful validation would be necessary in a real deployment scenario.